## Executive Summary

**Purpose:** the part of a winding outside the stack. Its end turns set the length of copper, and so the **phase resistance**, and they carry the **end-winding leakage** inductance. Phases whose end turns differ in length or radius, such as one phase per layer, get different resistances.

**What it does:**
- `EndTurn` describes the end-turn shape: a straight extension out of the slot, an optional axial tier offset for each slot position, an arc across the coil span, and an empirical leakage factor.
- `coil_end_length` and `mean_turn_length` give the copper length of each coil and phase.
- `phase_resistance` gives the DC resistance of each phase at a temperature, from either a copper area per turn or a fill factor.
- `end_winding_inductance` gives the empirical end-winding leakage of each phase.

**Background:** the end-winding leakage path is shown on the [Inductance Theory](inductance_theory.html) page.

**Assumptions:** DC resistance (no skin or proximity effect). The end turns follow the simple shape below. The end-winding leakage uses an empirical permeance factor, to be replaced by a filament (Biot–Savart) model later.

## How It Fits Into the Motor Design Workflow

**Upstream:** `WindingPattern` built from coils (`custom` or a builder; a slot table has no coils), and `StatorGeometry` (stack length, slot regions for the radius and area of each position).

**Downstream:** `StatorInductances` and `compare_windings` ([Total Stator Inductance](inductance_stator.html)), copper loss, the phase resistance in motor models.

## Theory

![Left: side view of one coil leaving the stack. Each conductor runs straight out of the slot for l_ext, plus a tier offset for its layer, then turns across the coil span. Layers 1, 2 and 3 bend at different axial distances so they can cross. Right: end view, the arc across the span of the coil at the mean radius of the conductors.](images/end_turn_geometry.svg)

### Step 1 — length of one end turn

A coil leaves the slot, runs axially for $l_{ext}$, crosses over to the other side's slot, and comes back. Where conductors of different layers have to cross, each layer bends at its own axial distance: the **tier** offset $\Delta_{pos}$. So the length of one end turn is

$$l_{end} = 2\,(l_{ext} + \Delta_{pos}) + k_{arc}\, \frac{2\pi\, r_{mid}}{Q}\, s,$$

with $s$ the coil span in slots (the shorter way round), $r_{mid}$ the mean radius of the coil's conductors (deeper layers sit at a larger radius, so their arc is longer), and $k_{arc}$ a shape factor (1 for a circular arc, about $1/\cos\alpha$ for a diamond end turn at angle $\alpha$).

The mean turn length of a coil is $\text{MLT} = 2\,l_{stk} + 2\,l_{end}$.

**Which layer gets which tier depends on how the winding is made.** If the bottom layer is inserted first it usually bends nearest the core, so the layers further out (towards the air gap) bend further away. Both orders are worked through in the example below.

### Step 2 — phase resistance

Each turn of a coil has copper area $A_c$. A phase with $a$ parallel paths has $1/a$ of its turns in each path and $a$ paths in parallel:

$$R_x = \frac{\rho(T)}{a^2} \sum_{\text{coils of } x} \frac{N_c\, \text{MLT}_c}{A_c}, \qquad \rho(T) = \rho_{20}\,\bigl(1 + \alpha\,(T - 20\,^\circ\text{C})\bigr).$$

$A_c$ is either given, or set by a copper fill factor $k_{fill}$: $A_c = k_{fill}\, A_{pos} / N_{pos}$, the copper share of the position's area divided by the turns in it (the smaller value of the coil's two sides). Copper: $\rho_{20} = 1.724 \times 10^{-8}\ \Omega\,\text{m}$, $\alpha = 3.93 \times 10^{-3}$ /K.

### Step 3 — end-winding leakage (empirical)

The end turns are in air, and the field around them is three-dimensional. The usual design formula treats each end as a conductor bundle of length $l_{end}$ with a permeance per unit length $\mu_0 \lambda_w$. A phase has $p$ coil groups of $N_s/p$ turns in series, each with two ends, so

$$L_{ew} = 2\,\mu_0\, \frac{N_s^2}{p}\, l_{end}\, \lambda_w.$$

$\lambda_w$ is empirical: about 0.2–0.5 depending on the winding type and on how close the end turns are to the core and frame (Pyrhönen et al., §4.3; Lipo, ch. 6). The default is 0.3. The factor is fitted to measured *synchronous* inductances, so it already includes the coupling between phases at the ends. The result is therefore placed on the diagonal, and it adds directly to $L_s$. Use it for estimates and for comparing windings on the same end-turn shape, and calibrate $\lambda_w$ against a measurement when you have one. Note: `StatorGeometry.l_ew` (the overhang drawn in the geometry figure) is a separate dimension; `EndTurn` carries the end-turn shape used here.

In [ ]:
#| default_exp inductance.endwinding

In [ ]:
#| hide
from dataclasses import replace
from emachines.inductance.geometry import SlotGeometry
from emachines.inductance.airgap import airgap_inductance, phase_imbalance
from emachines.inductance.slot import slot_leakage_inductance

In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from emachines.inductance.airgap import MU0
from emachines.inductance.geometry import StatorGeometry
from emachines.winding.pattern import Coil, WindingPattern

RHO_CU_20 = 1.724e-8      # copper resistivity at 20 °C [Ω·m] (IEC 60028, annealed)
ALPHA_CU = 3.93e-3        # copper temperature coefficient at 20 °C [1/K]

In [ ]:
#| export
@dataclass(frozen=True)
class EndTurn:
    """
    Shape of the end turns, for turn length and end-winding leakage.

    One end turn (one end of a coil) = 2 axial legs + an arc across the coil span:

        l_end = 2 (l_ext + tier[pos]) + k_arc · span · 2π r_mid / Q

    Attributes
    ----------
    l_ext     : float          Straight axial extension out of the slot before the turn [m]
    tier      : dict           Extra axial offset of each slot position [m], for end turns that
                               stack axially (e.g. {'1': 16e-3, '2': 8e-3, '3': 0.0}); default none
    k_arc     : float          Arc length factor: 1 = circular arc at the conductors' mean
                               radius; > 1 for diamond or involute end turns (default 1)
    lambda_ew : float          Empirical end-winding permeance factor λ_w (default 0.3)
    """
    l_ext: float
    tier: dict = field(default_factory=dict)
    k_arc: float = 1.0
    lambda_ew: float = 0.3

    def __post_init__(self):
        if self.l_ext < 0 or self.k_arc <= 0 or self.lambda_ew < 0:
            raise ValueError("l_ext and lambda_ew must not be negative and k_arc must be positive")
        if any(v < 0 for v in self.tier.values()):
            raise ValueError("tier offsets must not be negative")


def _coils(pattern: WindingPattern) -> tuple:
    if not pattern.coils:
        raise ValueError(f"{pattern.name or 'this pattern'} has no coils (built from a slot table?); "
                         "end-turn lengths need the coils, so build it with custom() or a builder")
    return pattern.coils


def coil_end_length(coil: Coil, pattern: WindingPattern, stator: StatorGeometry, end: EndTurn) -> float:
    """
    Length of one end turn of a coil [m].

    The span is the shorter way round between the two slots. The arc is taken at the mean radius
    of the two sides' regions; the legs use the mean tier offset of the two positions.
    """
    regs = {r.name: r for r in stator.regions(pattern.layout, len(pattern.positions))}
    r_mid = stator.r_body + np.mean([(regs[p].y0 + regs[p].y1) / 2 for p in (coil.out_pos, coil.in_pos)])
    d = (coil.in_slot - coil.out_slot) % pattern.Q
    span = min(d, pattern.Q - d)
    tier = np.mean([end.tier.get(p, 0.0) for p in (coil.out_pos, coil.in_pos)])
    return float(2 * (end.l_ext + tier) + end.k_arc * span * 2 * np.pi * r_mid / pattern.Q)


def mean_turn_length(pattern: WindingPattern, stator: StatorGeometry, end: EndTurn) -> np.ndarray:
    """
    Mean turn length of each phase [m], turn-weighted over its coils:
    MLT = 2 l_stk + 2 l_end.

    Returns
    -------
    np.ndarray, shape (m,)
    """
    total, turns = np.zeros(pattern.m), np.zeros(pattern.m)
    for c in _coils(pattern):
        total[c.phase] += c.turns * (2 * stator.l_stk + 2 * coil_end_length(c, pattern, stator, end))
        turns[c.phase] += c.turns
    return total / np.where(turns > 0, turns, 1)

In [ ]:
#| export
def phase_resistance(pattern: WindingPattern, stator: StatorGeometry, end: EndTurn,
                     a_cond: float | None = None, fill: float | None = None, T: float = 20.0,
                     rho20: float = RHO_CU_20, alpha: float = ALPHA_CU) -> np.ndarray:
    """
    DC resistance of each phase [Ω].

    R_x = ρ(T) Σ_coils (turns · MLT_coil / A_coil) / a², with a the parallel paths and
    ρ(T) = ρ20 (1 + α (T − 20)).

    Parameters
    ----------
    pattern, stator, end
    a_cond : float | None   Copper area of one turn [m²], the same for every coil; or
    fill   : float | None   Copper fill factor of the slot positions: each coil gets
                            fill × area of its position / turns at that position in the slot
                            (the smaller of its two sides)
    T      : float          Conductor temperature [°C] (default 20)
    rho20, alpha : float    Resistivity at 20 °C and its temperature coefficient (copper)

    Returns
    -------
    np.ndarray, shape (m,)
    """
    if (a_cond is None) == (fill is None):
        raise ValueError("give exactly one of a_cond (copper area per turn) or fill (copper fill factor)")
    if (a_cond is not None and a_cond <= 0) or (fill is not None and not 0 < fill <= 1):
        raise ValueError("a_cond must be positive and fill must be in (0, 1]")
    rho = rho20 * (1 + alpha * (T - 20.0))
    regs = {r.name: r for r in stator.regions(pattern.layout, len(pattern.positions))}
    idx = {p: k for k, p in enumerate(pattern.positions)}
    turns_at = np.abs(pattern.slot_turns).sum(axis=0)                  # (Q, positions), all phases
    R = np.zeros(pattern.m)
    for c in _coils(pattern):
        if a_cond is not None:
            area = a_cond
        else:
            assert fill is not None
            sides = []
            for slot, pos in ((c.out_slot, c.out_pos), (c.in_slot, c.in_pos)):
                reg = regs[pos]
                a_pos = stator.slot.area(stator.Q, reg.y0, reg.y1) * (reg.x1 - reg.x0)
                sides.append(fill * a_pos / turns_at[slot, idx[pos]])
            area = min(sides)
        mlt = 2 * stator.l_stk + 2 * coil_end_length(c, pattern, stator, end)
        R[c.phase] += rho * c.turns * mlt / area
    return R / pattern.parallel_paths ** 2


def end_winding_inductance(pattern: WindingPattern, stator: StatorGeometry, end: EndTurn) -> np.ndarray:
    """
    Empirical end-winding leakage inductance [H], as a diagonal matrix.

    L_ew,x = 2 μ0 (N_s,x² / p) · l_end,x · λ_w

    with N_s,x the series turns of phase x, l_end,x its mean end-turn length (one end) and
    λ_w = ``end.lambda_ew``. The factor λ_w is empirical and already includes the coupling
    between phases at the ends, so the result goes on the diagonal and adds directly to the
    synchronous inductance. Calibrate λ_w against a measurement where you can.

    Returns
    -------
    np.ndarray, shape (m, m)
    """
    l_sum = np.zeros(pattern.m)
    turns = np.zeros(pattern.m)
    for c in _coils(pattern):
        l_sum[c.phase] += c.turns * coil_end_length(c, pattern, stator, end)
        turns[c.phase] += c.turns
    l_end = l_sum / np.where(turns > 0, turns, 1)
    Ns = pattern.series_turns()
    return np.diag(2 * MU0 * Ns ** 2 / pattern.p * l_end * end.lambda_ew)

---

## Examples

The 36-slot stator of the earlier notebooks, a short-pitched double layer with 10 turns per coil and 2 parallel paths, at 20 °C and 120 °C. The fill factor is 0.42:

In [ ]:
stator = StatorGeometry(Q=36, r_bore=60e-3, g=0.8e-3, l_stk=100e-3, h_y=12e-3, l_ew=22e-3,
                        slot=SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5))
pat = WindingPattern.distributed(36, 4, pitch=7, turns=10, parallel_paths=2)
end = EndTurn(l_ext=12e-3)
print(f"one end turn {coil_end_length(pat.coils[0], pat, stator, end) * 1e3:.1f} mm, "
      f"MLT {mean_turn_length(pat, stator, end)[0] * 1e3:.1f} mm")
for T in (20, 120):
    print(f"R at {T:3d} °C: {np.round(phase_resistance(pat, stator, end, fill=0.42, T=T) * 1e3, 2)} mΩ")
Lew = end_winding_inductance(pat, stator, end)
print(f"end-winding leakage {Lew[0, 0] * 1e3:.4f} mH per phase "
      f"(λ_w = {end.lambda_ew}), air-gap synchronous {phase_imbalance(airgap_inductance(pat, stator))['L_positive'] * 1e3:.3f} mH")

### Resistance imbalance of a winding with one phase per layer (EP 2 237 392 B1)

The patent's 48-slot winding puts each phase in its own layer (dimensions assumed, as in the slot-leakage notebook). The deeper layers sit at a larger radius, so their arcs are longer. How the end turns stack axially then decides which phase gets the longest legs. Three cases at 120 °C, fill 0.45:

In [ ]:
def ep2237392(order=("1", "2", "3"), turns=2):
    """EP 2 237 392 B1: 48 slots, 4 poles, one phase per radial layer (phase k in layer order[k])."""
    coils = []
    for ph, (start, sign0) in enumerate([(0, 1), (4, -1), (8, 1)]):
        for pole in range(4):
            s0, sign = start + 12 * pole, sign0 * (-1) ** pole
            pos = order[ph]
            for k in range(6):                                   # concentric: spans 1, 3, ..., 11
                a, b = (s0 + 5 - k) % 48, (s0 + 6 + k) % 48
                if sign < 0:
                    a, b = b, a
                coils.append(Coil(ph, a, pos, b, pos, turns))
    return WindingPattern.custom(48, 4, coils, layout="over_under", n_layers=3,
                                 name="EP2237392")

ep_stator = StatorGeometry(Q=48, r_bore=80e-3, g=0.6e-3, l_stk=150e-3, h_y=15e-3, l_ew=30e-3,
                           slot=SlotGeometry.open(5e-3, 24e-3))

In [ ]:
ep = ep2237392()
cases = {"radius only, no tiers": {},
         "bottom layer nearest core (8 mm tiers)": {"1": 16e-3, "2": 8e-3, "3": 0.0},
         "top layer nearest core (8 mm tiers)": {"1": 0.0, "2": 8e-3, "3": 16e-3}}
for label, tier in cases.items():
    e = EndTurn(l_ext=12e-3, tier=tier)
    R = phase_resistance(ep, ep_stator, e, fill=0.45, T=120)
    mlt = mean_turn_length(ep, ep_stator, e)
    Lew = np.diag(end_winding_inductance(ep, ep_stator, e))
    print(f"{label:40s} MLT A/B/C {np.round(mlt * 1e3, 1)} mm   R {np.round(R * 1e3, 1)} mΩ  "
          f"spread {100 * (R.max() - R.min()) / R.mean():4.1f}%   L_ew {np.round(Lew * 1e6, 1)} µH")

With no tiers, the radius alone gives a 5% resistance spread, with phase C (at the slot bottom) longest. Tiers that put the bottom layer nearest the core shorten C's legs and lengthen A's, reversing the order. Tiers the other way add to the radius effect, and the spread grows to about 17%. The patent does not say which is used, so the real spread is somewhere in this range.

---

## Validation

1. **Turn length by hand.** For a single-layer full-pitch winding with no tiers, every end turn is $2 l_{ext} + (Q/P)\,2\pi r_{mid}/Q$, with $r_{mid}$ at the middle of the slot body.
2. **Resistance by hand.** With a given copper area $A_c$ and equal coils, $R = \rho\, N_{coils} N_c\, \text{MLT} / (A_c\, a^2)$ per phase. Halving the area doubles $R$, two paths divide it by 4, and $R(T)/R(20) = 1 + \alpha(T - 20)$.
3. **End-winding leakage.** It scales as $N_s^2$ and $l_{end}$, and is the closed form $2\mu_0 N_s^2 l_{end} \lambda_w / p$.

In [ ]:
st1 = StatorGeometry(Q=24, r_bore=50e-3, g=0.6e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(5e-3, 20e-3))
p1 = WindingPattern.distributed(24, 4, layers=1, turns=8)
e1 = EndTurn(l_ext=10e-3)
r_mid = st1.r_body + 10e-3
l_end = 2 * 10e-3 + 6 * 2 * np.pi * r_mid / 24
assert all(np.isclose(coil_end_length(c, p1, st1, e1), l_end) for c in p1.coils)
mlt = 2 * 80e-3 + 2 * l_end
assert np.allclose(mean_turn_length(p1, st1, e1), mlt)
R = phase_resistance(p1, st1, e1, a_cond=1e-6)
n_coils = len(p1.coils) // 3
assert np.allclose(R, RHO_CU_20 * n_coils * 8 * mlt / 1e-6)
print(f"by hand: l_end = {l_end * 1e3:.2f} mm, MLT = {mlt * 1e3:.2f} mm, R = {R[0] * 1e3:.3f} mΩ")
Lew = end_winding_inductance(p1, st1, e1)
assert np.allclose(np.diag(Lew), 2 * MU0 * p1.series_turns() ** 2 / 2 * l_end * 0.3)

---

## Tests

In [ ]:
#| hide
# scaling: area, paths, temperature, turns
assert np.allclose(phase_resistance(p1, st1, e1, a_cond=0.5e-6), 2 * R)
p2 = WindingPattern.distributed(24, 4, layers=1, turns=8, parallel_paths=2)
assert np.allclose(phase_resistance(p2, st1, e1, a_cond=1e-6), R / 4)
assert np.allclose(phase_resistance(p1, st1, e1, a_cond=1e-6, T=120), R * (1 + ALPHA_CU * 100))
assert np.allclose(end_winding_inductance(WindingPattern.distributed(24, 4, layers=1, turns=16), st1, e1), 4 * Lew)
# fill factor: single layer, one coil side per slot -> A_c = fill * slot area / turns
assert np.allclose(phase_resistance(p1, st1, e1, fill=0.5),
                   phase_resistance(p1, st1, e1, a_cond=0.5 * st1.slot.area(24) / 8))
# balanced windings: equal resistance and end leakage
for p_ in [WindingPattern.distributed(36, 4, pitch=7), WindingPattern.concentrated(12, 10)]:
    st_ = StatorGeometry(Q=p_.Q, r_bore=55e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(5e-3, 15e-3))
    R_ = phase_resistance(p_, st_, e1, fill=0.4)
    assert np.allclose(R_, R_[0]) and np.allclose(np.diag(end_winding_inductance(p_, st_, e1)), end_winding_inductance(p_, st_, e1)[0, 0])
# tiers lengthen the legs; k_arc scales the arc only
e_t = EndTurn(l_ext=10e-3, tier={"full": 5e-3})
assert np.isclose(coil_end_length(p1.coils[0], p1, st1, e_t) - l_end, 2 * 5e-3)
assert np.isclose(coil_end_length(p1.coils[0], p1, st1, EndTurn(10e-3, k_arc=1.2)) - l_end, 0.2 * (l_end - 20e-3))
# EP: radius effect puts the deepest layer (phase C) longest
mlt_ep = mean_turn_length(ep, ep_stator, EndTurn(12e-3))
assert mlt_ep[0] < mlt_ep[1] < mlt_ep[2]
# errors
for f in [lambda: phase_resistance(p1, st1, e1), lambda: phase_resistance(p1, st1, e1, a_cond=1e-6, fill=0.4),
          lambda: phase_resistance(p1, st1, e1, fill=1.5), lambda: EndTurn(-1e-3),
          lambda: mean_turn_length(WindingPattern.from_slot_table([[1, -3, 2, -1, 3, -2]], 2), st1, e1)]:
    try:
        f(); raise AssertionError("expected ValueError")
    except ValueError:
        pass

## References

- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley, §4.3 (end-winding leakage factor) and §2 (winding geometry).
- Lipo, T.A. (2017). *Introduction to AC Machine Design*. Wiley-IEEE Press, ch. 6 — end-winding leakage, end-turn length.
- IEC 60028 — resistivity of annealed copper, 1/58 Ω mm²/m at 20 °C.
- EP 2 237 392 B1, *AC motor winding pattern* — the one-phase-per-layer winding of the example (dimensions assumed).